# 04. 체인(Chain)과 LCEL 추적

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LCEL 체인(`|` 연산자)이 LangSmith에서 계층적 부모-자식 Trace로 기록되는 구조를 설명할 수 있어요
2. `RunnablePassthrough`, `RunnableParallel`의 Trace 표현 방식을 이해하고 해석할 수 있어요
3. `RunnableLambda`를 사용한 커스텀 함수가 Trace에 어떻게 나타나는지 확인할 수 있어요
4. 오류 발생 시 LangSmith Trace에서 실패 지점을 빠르게 찾아낼 수 있어요

## 사전 지식

- `03-Chat-Model-Tracing` 노트북 완료
- LCEL(LangChain Expression Language) 기본 사용법 (`|` 연산자)
- `ChatPromptTemplate`, `StrOutputParser` 사용 경험

## 개념 설명: LCEL Trace 계층 구조

LCEL 체인(`|` 연산자로 연결)은 실행될 때 각 컴포넌트가 독립적인 Span으로 기록돼요. 이 Span들은 계층(부모-자식) 구조를 이루어요.

### 기본 LCEL 체인 Trace 구조

```
chain (RunnableSequence) ← 루트 Trace
  ├── ChatPromptTemplate  ← 자식 Span 1
  ├── ChatOpenAI          ← 자식 Span 2 (run_type=llm)
  └── StrOutputParser     ← 자식 Span 3
```

### 계층 구조 다이어그램

```mermaid
flowchart TD
    A[사용자 입력] --> B[chain.invoke]
    B --> C[ChatPromptTemplate]
    C --> D[ChatOpenAI]
    D --> E[StrOutputParser]
    E --> F[최종 출력]

    B --> G[LangSmith: RunnableSequence]
    G --> H[Span: ChatPromptTemplate]
    G --> I[Span: ChatOpenAI - run_type=llm]
    G --> J[Span: StrOutputParser]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C,D,E process
    class G,H,I,J storage
    class F output
```

### 핵심 구성 요소

| LCEL 컴포넌트 | LangSmith에서의 Span 이름 | 특징 |
|--------------|--------------------------|------|
| `ChatPromptTemplate` | `ChatPromptTemplate` | 입출력: 변수 → 프롬프트 메시지 |
| `ChatOpenAI` | `ChatOpenAI` | `run_type=llm`, 토큰/비용 포함 |
| `StrOutputParser` | `StrOutputParser` | 입출력: AIMessage → 문자열 |
| `RunnablePassthrough` | `RunnablePassthrough` | 입력을 그대로 통과 |
| `RunnableParallel` | `RunnableParallel` | 병렬 자식 Span 생성 |
| `RunnableLambda` | 함수 이름 | 커스텀 함수 래핑 |

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경변수 로드 및 프로젝트 설정
# ---------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

# 이 노트북 전용 LangSmith 프로젝트
os.environ["LANGSMITH_PROJECT"] = "Ch04-Chain-LCEL-Tracing"

print("프로젝트:", os.environ["LANGSMITH_PROJECT"])

## 1. 기본 LCEL 체인 추적

가장 기본적인 LCEL 체인부터 시작해요. `prompt | model | parser` 형태의 체인이 LangSmith에서 어떻게 기록되는지 확인해요.

> 🎯 **강의 포인트**: 체인 실행 후 LangSmith UI에서 루트 Trace를 클릭하면 왼쪽에 트리 구조로 각 단계가 표시돼요. 각 단계의 입출력이 어떻게 변환되는지 직접 확인해 보세요.

In [2]:
# ---------------------------------------------------
# 기본 LCEL 체인 구성
# ---------------------------------------------------
# 3단계 체인: 프롬프트 → 모델 → 파서
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

# 각 컴포넌트 정의
prompt = ChatPromptTemplate.from_template(
    "다음 주제에 대해 {length}로 설명해 주세요: {topic}"
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()  # AIMessage → 문자열 변환

# LCEL 체인 구성: | 연산자로 연결해요
chain = prompt | model | parser

# 체인 실행
result = chain.invoke({
    "topic": "임베딩 벡터",
    "length": "2문장"
})

print("결과:", result)
print("\nLangSmith UI에서 Trace 트리 구조를 확인해 보세요!")

결과: 임베딩 벡터는 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 특징을 추출하는 방법입니다. 주로 자연어 처리와 추천 시스템에서 사용되며, 단어, 문장, 또는 아이템 간의 유사성을 수치적으로 표현하는 데 활용됩니다.

LangSmith UI에서 Trace 트리 구조를 확인해 보세요!


### LangSmith에서 확인할 내용

1. **Traces 뷰** vs **Runs 뷰** 차이:
   - **Traces 뷰**: 루트 Trace만 표시 (체인 전체)
   - **Runs 뷰**: 모든 Span 표시 (각 단계 포함)

> 🔑 **핵심 개념**: **Traces 뷰**는 "대화 하나"를 보고 싶을 때, **Runs 뷰**는 특정 단계(예: LLM 호출만)를 분석할 때 사용해요.

## 2. RunnablePassthrough와 RunnableParallel 추적

`RunnablePassthrough`는 입력을 그대로 다음 단계로 전달해요. `RunnableParallel`은 여러 경로를 병렬로 실행해요.

> 💡 **실무 팁**: `RunnableParallel`은 LangSmith에서 동일한 레벨에 여러 자식 Span이 나란히 표시돼요. 각 병렬 경로의 처리 시간을 비교해 병목을 찾을 수 있어요.

In [3]:
# ---------------------------------------------------
# RunnablePassthrough: 입력 그대로 전달
# ---------------------------------------------------
# RAG 패턴에서 자주 사용해요: context는 검색, question은 그대로 전달
from langchain.schema.runnable import RunnablePassthrough

# 컨텍스트를 반환하는 더미 리트리버
def get_context(question: str) -> str:
    """실제 RAG에서는 벡터 DB 검색이 여기서 이루어져요"""
    return f"[검색된 컨텍스트] {question}에 관한 문서 내용입니다."

# RAG 체인 패턴: context는 리트리버, question은 그대로 통과
rag_prompt = ChatPromptTemplate.from_template(
    "컨텍스트: {context}\n\n질문: {question}\n\n답변:"
)

# RunnablePassthrough로 질문을 그대로 전달하면서 context도 제공
rag_chain = (
    {
        "context": lambda x: get_context(x["question"]),  # 리트리버
        "question": RunnablePassthrough()                   # 질문은 그대로 통과
    }
    | rag_prompt
    | model
    | parser
)

result = rag_chain.invoke({"question": "벡터 데이터베이스란??"})
print("RAG 체인 결과:", result[:200])

RAG 체인 결과: 벡터 데이터베이스는 고차원 벡터를 저장하고 검색하는 데 최적화된 데이터베이스입니다. 이러한 데이터베이스는 주로 머신러닝 및 인공지능 분야에서 사용되며, 텍스트, 이미지, 오디오 등 다양한 형태의 데이터를 벡터로 변환하여 저장합니다. 벡터 데이터베이스는 유사성 검색, 즉 특정 벡터와 유사한 벡터를 빠르게 찾는 기능을 제공합니다. 이를 통해 추천 시스템, 이미


In [4]:
# ---------------------------------------------------
# RunnableParallel: 여러 경로 병렬 실행
# ---------------------------------------------------
# 동일한 입력으로 여러 작업을 동시에 처리해요
from langchain.schema.runnable import RunnableParallel

# 두 가지 분석을 병렬로 실행하는 체인
summary_prompt = ChatPromptTemplate.from_template("{text}를 한 줄로 요약해 주세요.")
keyword_prompt = ChatPromptTemplate.from_template("{text}의 핵심 키워드 3개를 뽑아 주세요.")

# 두 체인을 병렬로 구성해요
parallel_chain = RunnableParallel(
    summary=summary_prompt | model | parser,   # 요약 경로
    keywords=keyword_prompt | model | parser   # 키워드 경로
)

text = "LangChain은 대규모 언어 모델을 활용한 애플리케이션 개발 프레임워크예요. 다양한 모델, 도구, 메모리 컴포넌트를 체인으로 연결할 수 있어요."

result = parallel_chain.invoke({"text": text})

print("=== 병렬 처리 결과 ===")
print("요약:", result["summary"])
print("키워드:", result["keywords"])
print("\nLangSmith에서 두 경로가 나란히 실행된 Span을 확인해 보세요!")

=== 병렬 처리 결과 ===
요약: LangChain은 대규모 언어 모델을 활용하여 다양한 모델과 도구를 체인으로 연결해 애플리케이션을 개발할 수 있는 프레임워크입니다.
키워드: 1. 대규모 언어 모델
2. 애플리케이션 개발
3. 체인 연결

LangSmith에서 두 경로가 나란히 실행된 Span을 확인해 보세요!


## 3. RunnableLambda로 커스텀 함수 추적

`RunnableLambda`는 일반 Python 함수를 LCEL 체인에 통합해요. LangSmith에서 함수 이름으로 Span이 생성돼요.

> 🔑 **핵심 개념**: `RunnableLambda`를 사용하면 커스텀 전처리/후처리 함수가 Trace에 명시적으로 나타나요. 디버깅 시 어떤 함수에서 데이터가 변환되는지 추적하기 쉬워요.

In [5]:
# ---------------------------------------------------
# RunnableLambda: 커스텀 함수를 체인에 통합
# ---------------------------------------------------
# 일반 Python 함수를 LCEL 체인 단계로 만들어요
from langchain.schema.runnable import RunnableLambda

def preprocess_input(text: str) -> str:
    """입력 전처리: 공백 제거 + 소문자 변환 (예시)"""  
    return text.strip().lower()

def postprocess_output(text: str) -> dict:
    """출력 후처리: 문자열을 구조화된 딕셔너리로 변환"""
    return {
        "answer": text,
        "word_count": len(text.split()),
        "char_count": len(text)
    }

# 전처리 → 프롬프트 → 모델 → 파서 → 후처리 체인
full_chain = (
    RunnableLambda(preprocess_input)  # 전처리 단계 (LangSmith에서 'preprocess_input' Span으로 표시)
    | ChatPromptTemplate.from_template("다음 질문에 답해 주세요: {input}")
    | model
    | parser
    | RunnableLambda(postprocess_output)  # 후처리 단계
)

result = full_chain.invoke("  LLM이란 무엇인가요?  ")  # 앞뒤 공백 포함

print("결과:")
print(f"  답변: {result['answer'][:150]}...")
print(f"  단어 수: {result['word_count']}")
print(f"  글자 수: {result['char_count']}")

결과:
  답변: LLM은 "Large Language Model"의 약자로, 대규모 언어 모델을 의미합니다. 이러한 모델은 인공지능(AI) 기술의 일종으로, 방대한 양의 텍스트 데이터를 학습하여 자연어를 이해하고 생성하는 능력을 갖추고 있습니다. LLM은 주로 딥러닝 기술을 기반으로 ...
  단어 수: 75
  글자 수: 426


## 4. 오류 발생 시 Trace 분석

체인 실행 중 오류가 발생하면 LangSmith에서 정확히 어느 단계에서 실패했는지 확인할 수 있어요.

> ⚠️ **자주 하는 실수**: 오류가 발생했을 때 Python 스택 트레이스만 보지 말고, LangSmith Trace에서 실패한 Span(빨간색 표시)을 클릭해서 해당 단계의 **입력값**을 확인하세요. 종종 데이터 형식 문제가 원인이에요.

In [6]:
# ---------------------------------------------------
# 오류 처리 예제: 실패 지점 Trace 확인
# ---------------------------------------------------
# 의도적으로 오류를 발생시켜 LangSmith에서 실패 지점을 확인해요

def risky_formatter(data: dict) -> str:
    """위험한 함수: 특정 조건에서 오류 발생 (예시)"""
    # 'message' 키가 없으면 오류 발생
    return data["message"]  # KeyError 발생 가능

# 오류 발생 가능성이 있는 체인
risky_chain = (
    RunnableLambda(risky_formatter)
    | ChatPromptTemplate.from_template("{input}")
    | model
    | parser
)

# 오류 발생: 'message' 키가 없는 딕셔너리 전달
try:
    result = risky_chain.invoke({"query": "이 키는 없어요"})  # 'message' 키 없음
except Exception as e:
    print(f"오류 발생: {type(e).__name__}: {e}")
    print("\nLangSmith UI에서:")
    print("  1. 프로젝트 페이지로 이동해요")
    print("  2. 실패한 Trace를 찾아요 (빨간색 아이콘)")
    print("  3. 'risky_formatter' Span을 클릭해요")
    print("  4. Error 탭에서 스택 트레이스와 입력값을 확인해요")

오류 발생: KeyError: 'message'

LangSmith UI에서:
  1. 프로젝트 페이지로 이동해요
  2. 실패한 Trace를 찾아요 (빨간색 아이콘)
  3. 'risky_formatter' Span을 클릭해요
  4. Error 탭에서 스택 트레이스와 입력값을 확인해요


In [7]:
# ---------------------------------------------------
# 수정된 체인: 오류 처리 추가
# ---------------------------------------------------
# 오류 원인을 파악한 후 수정한 버전이에요

def safe_formatter(data: dict) -> str:
    """안전한 함수: .get()으로 기본값 처리"""
    # 'message' 키가 없으면 'query' 키를 시도하고, 그것도 없으면 기본값 사용
    return data.get("message") or data.get("query", "기본 질문")

safe_chain = (
    RunnableLambda(safe_formatter)
    | ChatPromptTemplate.from_template("{input}")
    | model
    | parser
)

# 이번엔 성공해요
result = safe_chain.invoke({"query": "임베딩이란 무엇인가요?"})
print("안전한 체인 결과:", result[:200])

안전한 체인 결과: 임베딩(embedding)은 고차원 데이터를 저차원 공간에 매핑하는 방법을 의미합니다. 주로 자연어 처리(NLP), 이미지 처리, 추천 시스템 등 다양한 분야에서 사용됩니다. 임베딩의 주요 목적은 데이터의 의미를 보존하면서 더 간결하고 효율적인 표현을 만드는 것입니다.

예를 들어, 자연어 처리에서 단어 임베딩은 단어를 고차원 벡터 공간의 점으로 변환하여 


## 5. 중간 단계 입출력 확인

LCEL 체인의 각 단계에서 데이터가 어떻게 변환되는지 코드와 LangSmith에서 동시에 확인해요.

> 💡 **실무 팁**: `chain.invoke()` 대신 각 단계를 개별적으로 실행해서 중간 출력을 확인할 수 있어요. 하지만 LangSmith를 사용하면 전체 체인을 한 번에 실행하면서도 모든 중간 단계를 UI에서 확인할 수 있어요.

In [8]:
# ---------------------------------------------------
# 중간 단계 입출력 직접 확인
# ---------------------------------------------------
# 각 단계를 순서대로 실행해서 변환 과정을 보여줘요

from langchain.schema import HumanMessage

# 분석할 데이터
input_data = {"topic": "FAISS", "length": "3문장"}

# 1단계: 프롬프트 변환 결과 확인
formatted_prompt = prompt.invoke(input_data)
print("1. 프롬프트 변환 결과:")
print(f"   타입: {type(formatted_prompt).__name__}")
print(f"   메시지: {formatted_prompt.messages}")

# 2단계: 모델 호출 결과 확인
model_output = model.invoke(formatted_prompt)
print("\n2. 모델 출력 결과:")
print(f"   타입: {type(model_output).__name__}")
print(f"   내용: {model_output.content[:100]}...")

# 3단계: 파서 출력 확인
parsed_output = parser.invoke(model_output)
print("\n3. 파서 출력 결과:")
print(f"   타입: {type(parsed_output).__name__}")
print(f"   내용: {parsed_output[:100]}...")

print("\n각 단계의 Trace가 LangSmith에 별도 Span으로 기록됐어요!")

1. 프롬프트 변환 결과:
   타입: ChatPromptValue
   메시지: [HumanMessage(content='다음 주제에 대해 3문장로 설명해 주세요: FAISS', additional_kwargs={}, response_metadata={})]

2. 모델 출력 결과:
   타입: AIMessage
   내용: FAISS(Facebook AI Similarity Search)는 대규모 데이터셋에서 유사한 벡터를 효율적으로 검색하기 위한 라이브러리입니다. 이 라이브러리는 고속 검색을 위해 ...

3. 파서 출력 결과:
   타입: str
   내용: FAISS(Facebook AI Similarity Search)는 대규모 데이터셋에서 유사한 벡터를 효율적으로 검색하기 위한 라이브러리입니다. 이 라이브러리는 고속 검색을 위해 ...

각 단계의 Trace가 LangSmith에 별도 Span으로 기록됐어요!


## 6. 실습: 복합 체인 구성 및 Trace 분석

In [9]:
# ============================================================
# TODO: 아래 체인에 단계를 추가하고 LangSmith Trace를 분석해 보세요
# 힌트: RunnableLambda로 전처리 함수를 추가하거나, RunnableParallel로 병렬 처리를 추가해 보세요
# 예상 결과: 추가한 단계가 LangSmith Trace 트리에 새 Span으로 나타나요
# ============================================================

from langchain_core.tracers.langchain import wait_for_all_tracers

# 기본 체인 (여기에 단계를 추가해 보세요)
base_prompt = ChatPromptTemplate.from_template(
    "{input}에 대해 간단히 설명해 주세요."
)

# 직접 수정해 보세요: 예) 전처리, 후처리 추가
my_chain = base_prompt | model | parser

# 실행
test_input = "트랜스포머 아키텍처"
result = my_chain.invoke({"input": test_input})

print(f"입력: {test_input}")
print(f"결과: {result[:300]}")

# 모든 Trace 전송 완료
wait_for_all_tracers()
print("\nLangSmith UI에서 Trace 트리를 확인해 보세요!")

입력: 트랜스포머 아키텍처
결과: 트랜스포머 아키텍처는 자연어 처리(NLP) 분야에서 널리 사용되는 딥러닝 모델로, 2017년 "Attention is All You Need"라는 논문에서 처음 소개되었습니다. 이 아키텍처는 주로 다음과 같은 주요 구성 요소로 이루어져 있습니다.

1. **Self-Attention Mechanism**: 입력 데이터의 각 단어가 다른 단어와의 관계를 학습할 수 있도록 하는 메커니즘입니다. 이를 통해 문맥을 이해하고, 단어 간의 상관관계를 효과적으로 파악할 수 있습니다.

2. **Multi-Head Attention**: 여러 개

LangSmith UI에서 Trace 트리를 확인해 보세요!


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **계층적 Trace**: LCEL 체인의 각 컴포넌트가 부모-자식 Span 구조로 기록돼요
- **Traces vs Runs 뷰**: Traces 뷰는 루트 실행만, Runs 뷰는 모든 Span을 보여줘요
- **RunnablePassthrough**: 입력을 그대로 전달하는 Span으로 Trace에 나타나요
- **RunnableParallel**: 병렬 실행되는 자식 Span들이 동일 레벨에 나란히 표시돼요
- **RunnableLambda**: 일반 Python 함수를 Trace 가능한 Span으로 만들어요
- **오류 추적**: 실패한 Span은 빨간색으로 표시되어 정확한 실패 지점을 찾을 수 있어요

## 다음 노트북 예고

다음 `05-Memory-Tracing.ipynb`에서는 **멀티턴 대화의 추적 방법**을 배워요. `thread_id`로 여러 대화 턴을 하나의 세션으로 그룹핑하고, 대화 히스토리에 따른 토큰 증가 패턴을 LangSmith에서 관찰할게요.